# Chapter 4A — Tokenization: From Characters to Subwords
**Series: LM from First Principles**

---

## What this chapter covers

Chapter 3 gave every token ID a learned vector. But token IDs come from a tokenizer —
and we have not yet asked what a token should actually be.

So far, we have used characters as tokens. That worked — but is a character always the right unit?

```
  Raw text:   "To be or not to be"
       ↓  tokenizer (this chapter)
  Char:   [T][o][ ][b][e]...          → 18 tokens
  BPE:    [To][be][or][not][to][be]   → 6 tokens
       ↓  embedding table
  Vectors: (T, C) float32 tensor
       ↓  transformer
  Prediction over vocabulary
```

Chapter 3 began at token IDs. This chapter builds the missing step immediately before them.

We try three approaches — characters, words, and subwords — and apply the
**Byte Pair Encoding (BPE)** algorithm to learn the subword vocabulary.
BPE is a data compression algorithm repurposed for tokenization: given a corpus,
it repeatedly merges the most frequent adjacent symbol pair, so common sequences
like th or the earn their own token while rare sequences stay split.
Section 5 runs it on tinyshakespeare so we can watch the vocabulary emerge from
raw frequency statistics.

**Chapter 4A** produces a working tokenizer.
**Chapter 4B** asks whether it is a good one.


---
## How We Measure a Tokenizer (Chapter 4A)

One number matters here:

**Compression efficiency** — tokens produced ÷ characters consumed.

```
  "tokenization is fundamental"  (27 chars)

  Char:  t o k e n i z a t i o n  i s  f u n d a m e n t a l  → 27 tokens  ratio = 1.000
  BPE:   [token][ization][ ][is][ ][fundamental]               →  5 tokens  ratio = 0.185
```

Lower ratio = shorter sequences — the model sees more text in the same number of positions.

We track compression in a scorecard across this chapter. Two further metrics —
**morphological coherence** and **bits per character (BPC)** — require a trained model
and are introduced in Chapter 4B.


In [1]:
# input : nothing (setup cell)
# output: scorecard = {'char_tokenizer': {'compression': None}, 'bpe_500': {'compression': None}}
# Tracks only compression ratio in 4A.
# Morphological coherence and BPC are added in Chapter 4B.

scorecard = {
    "char_tokenizer": {"compression": None},
    "bpe_500":        {"compression": None},
}
print("Scorecard initialised (compression only).")
print(scorecard)


Scorecard initialised (compression only).
{'char_tokenizer': {'compression': None}, 'bpe_500': {'compression': None}}


In [2]:
# input : URL pointing to tinyshakespeare.txt
# output: text = 'First Citizen:\nBefore we proceed...'  (1,115,394 characters)
import re
import urllib.request
import math
from collections import Counter, defaultdict

# Download tinyshakespeare (same corpus as Chapter 1)
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "tinyshakespeare.txt")

with open("tinyshakespeare.txt", "r", encoding="utf-8") as f:
    text = f.read()

# Whitespace-normalize once: our BPE pre-tokenizer splits on whitespace,
# so original newlines and multi-spaces are discarded during tokenization.
# Comparing both tokenizers against normalized_text is apples-to-apples.
normalized_text = " ".join(text.split())

print(f"Corpus loaded: {len(text):,} characters")
print(f"Normalized  : {len(normalized_text):,} characters (whitespace-collapsed)")
print(f"First 80 chars: {repr(text[:80])}") 


Corpus loaded: 1,115,394 characters
Normalized  : 1,108,152 characters (whitespace-collapsed)
First 80 chars: 'First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.'


---
## Section 1 — The Character Tokenizer You Already Have

Chapter 3 used this corpus-derived character tokenizer: collect the characters present
in tinyshakespeare, sort them, and assign IDs. Vocabulary size: **65 tokens**.

Every single character in the input becomes exactly one token.


In [3]:
# input : text corpus  e.g. 'To be or not to be'
# output: stoi_char = {'\n':0, ' ':1, ...},  tokens_char = [32, 53, 1, ...]
# Rebuild the Chapter 1 character tokenizer
chars = sorted(set(text))
vocab_size_char = len(chars)
stoi_char = {ch: i for i, ch in enumerate(chars)}
itos_char = {i: ch for i, ch in enumerate(chars)}

encode_char = lambda s: [stoi_char[c] for c in s]
decode_char = lambda ids: ''.join(itos_char[i] for i in ids)

tokens_char = encode_char(text)

print(f"Vocabulary size  : {vocab_size_char} tokens")
print(f"Corpus tokens    : {len(tokens_char):,}")
print(f"Token/char ratio : {len(tokens_char)/len(text):.3f}  (exactly 1.0 — one char = one token)")
print()
print("Example encode:")
sample = "To be or not to be"
ids = encode_char(sample)
print(f"  '{sample}'")
print(f"  → {ids}")
print(f"  → {len(ids)} tokens for {len(sample)} characters")

Vocabulary size  : 65 tokens
Corpus tokens    : 1,115,394
Token/char ratio : 1.000  (exactly 1.0 — one char = one token)

Example encode:
  'To be or not to be'
  → [32, 53, 1, 40, 43, 1, 53, 56, 1, 52, 53, 58, 1, 58, 53, 1, 40, 43]
  → 18 tokens for 18 characters


**Observation:** 1,115,394 characters → 1,115,394 tokens. One-to-one. The vocabulary is small (65 unique characters).
This seems ideal — but the sequence length is the problem.


---
## Section 2 — The Sequence Length Problem

A language model has a finite **token budget** — the number of positions it processes in one
forward pass. The tokenizer controls how much text fits into those positions.

With **character tokenization**, a 512-token window covers ~512 characters — roughly 3–4
sentences of Shakespeare.

With **BPE tokenization**, the same 512-token window covers substantially more text,
because each token packages multiple characters. (BPE is the algorithm we run in
Section 5 — the coverage table below uses an illustrative ratio as a preview.)

The tokenizer is therefore a multiplier on context coverage.
Everything else held equal, a compressing tokenizer lets the model see more text
in the same number of positions.


In [4]:
# input : T values  e.g. [128, 256, 512, 1024, 2048]
# output: table showing chars covered per token budget for char vs BPE tokenizer
# A model has a fixed token budget T -- the number of positions it processes at once.
# The tokenizer determines how much text fits into that budget.
#
# char tokenizer : 1 token = 1 character  -> T tokens = T characters
# BPE tokenizer  : 1 token ~ 2.4 chars   -> T tokens ~ 2.4 * T characters
#
# Later, when we study how token positions communicate with each other,
# we will see that processing each position carries an additional cost that
# makes sequence length even more important. For now, coverage alone is enough.
# Illustrative ratio — Section 5 measures the exact value for our tokenizer.
# Using a round number here makes the table easy to read and reason about.
ILLUSTRATIVE_BPE_RATIO = 0.4   # illustrative; Section 5 measures the real value
AVG_SENTENCE_CHARS = 80  # approximate, for intuition only

print('Text covered per token budget (char vs BPE tokenizer):')
print(f"  {'T':>6}  {'Char tok (chars)':>20}  {'BPE tok (chars)':>18}  {'~BPE sentences'}")
print('  ' + '-'*65)
for T in [128, 256, 512, 1024, 2048]:
    chars_char = T
    chars_bpe  = int(T / ILLUSTRATIVE_BPE_RATIO)
    sentences  = chars_bpe / AVG_SENTENCE_CHARS
    print(f'  {T:>6}  {chars_char:>20,}  {chars_bpe:>18,}  (~{sentences:.0f} sentences)')
print()
print('Note: BPE ratio above is illustrative (~2.5x). Section 5 measures the exact value.')


Text covered per token budget (char vs BPE tokenizer):
       T      Char tok (chars)     BPE tok (chars)  ~BPE sentences
  -----------------------------------------------------------------
     128                   128                 320  (~4 sentences)
     256                   256                 640  (~8 sentences)
     512                   512               1,280  (~16 sentences)
    1024                 1,024               2,560  (~32 sentences)
    2048                 2,048               5,120  (~64 sentences)

Note: BPE ratio above is illustrative (~2.5x). Section 5 measures the exact value.


**Key takeaway:** For the same token budget T, BPE covers substantially more text
than character tokenization — the exact factor depends on the corpus and vocabulary size.
Section 5 will measure the precise compression ratio for our tokenizer.
The model sees a much richer context in the same number of token positions,
which matters for every task that requires tracking meaning across multiple sentences.


---
## Section 3 — The Other Extreme: Word Tokenization

If character tokenization makes sequences too long, what about splitting on whitespace?
One token per word. Sequences become short. But a new problem appears.

In [5]:
# input : text corpus  e.g. 'First Citizen: Before we proceed...'
# output: word_counts = {'the':5437, 'I':4403, ...},  ratio = 0.182 tokens/char
# Word tokenization: split on whitespace
words = text.split()
unique_words = set(words)

print(f"Total characters    : {len(text):,}")
print(f"Total word tokens   : {len(words):,}")
print(f"Unique words (vocab): {len(unique_words):,}")
print(f"Token/char ratio    : {len(words):,} / {len(text):,} = {len(words)/len(text):.3f}  (~1 token per 5 chars)")
print()

# Show most common words
word_counts = Counter(words)
print("20 most common words:")
for word, count in word_counts.most_common(20):
    print(f"  {word!r:20s} {count:6,}")

Total characters    : 1,115,394
Total word tokens   : 202,651
Unique words (vocab): 25,670
Token/char ratio    : 202,651 / 1,115,394 = 0.182  (~1 token per 5 chars)

20 most common words:
  'the'                 5,437
  'I'                   4,403
  'to'                  3,923
  'and'                 3,678
  'of'                  3,275
  'my'                  2,677
  'a'                   2,610
  'you'                 2,130
  'in'                  2,073
  'that'                1,812
  'And'                 1,801
  'is'                  1,768
  'not'                 1,631
  'with'                1,564
  'your'                1,493
  'be'                  1,489
  'his'                 1,391
  'for'                 1,381
  'have'                1,280
  'it'                  1,189


In [6]:
# input : word list + top-5000 vocabulary  e.g. ["thou'd", 'king', ...]
# output: per-word YES / OOV status  e.g. "thou'd" -> OOV, 'king' -> YES
# The OOV (out-of-vocabulary) problem
# Build a vocabulary from the top-5000 most common words
top5000 = {w for w, _ in word_counts.most_common(5000)}

# Now check related forms
test_words = ["thou", "thee", "thine", "thou'd", "thou'lt",
              "king", "kings", "kingly", "kingdom",
              "love", "loves", "loved", "lovest", "unlovable"]

print("OOV check (top-5000 word vocabulary):")
print(f"  {'Word':20s}  In vocab?")
print("  " + "-"*35)
for w in test_words:
    in_vocab = w in top5000
    print(f"  {w!r:20s}  {'YES' if in_vocab else 'OOV — unknown token'}")

print()
print("Words seen fewer than 5 times (long-tail problem):")
rare = [(w, c) for w, c in word_counts.items() if c < 5]
print(f"  {len(rare):,} unique words appear fewer than 5 times")
print(f"  That is {len(rare)/len(unique_words)*100:.1f}% of the vocabulary — almost all rare")

OOV check (top-5000 word vocabulary):
  Word                  In vocab?
  -----------------------------------
  'thou'                YES
  'thee'                YES
  'thine'               YES
  "thou'd"              OOV — unknown token
  "thou'lt"             OOV — unknown token
  'king'                YES
  'kings'               YES
  'kingly'              YES
  'kingdom'             YES
  'love'                YES
  'loves'               YES
  'loved'               YES
  'lovest'              YES
  'unlovable'           OOV — unknown token

Words seen fewer than 5 times (long-tail problem):
  21,417 unique words appear fewer than 5 times
  That is 83.4% of the vocabulary — almost all rare


**The word tokenization dilemma:**

- Vocabulary explodes: tens of thousands of unique words
- Most words are rare — the model barely sees them during training
- Morphological variants (`love`, `loves`, `loved`, `lovest`) get **separate embeddings** even though they share meaning
- New words at inference time are unknown

We need a middle ground — units larger than characters but smaller than words.

---
## Section 4 — The Middle Ground: Byte Pair Encoding (BPE)

BPE was originally a data compression algorithm (1994). In 2016, Sennrich et al. adapted it for neural machine translation. Today it underlies GPT-2, GPT-4, LLaMA, and most large language models.

**The algorithm:**
```
1. Start with the character vocabulary
2. Count every adjacent pair of tokens in the corpus
3. Find the most frequent pair
4. Merge that pair into a new single token
5. Replace all occurrences of the pair in the corpus with the new token
6. Repeat from step 2, N times
```

Each merge step creates one new vocabulary entry. After N unique merge products, the vocabulary consists of the base symbols (corpus characters + </w>) plus the learned merge symbols.

Let's trace it on a tiny example first.


### Step 1 — Represent the corpus

Before BPE can count anything, it needs the corpus in a form it can manipulate.  
Each word becomes a **tuple of characters** with a special end-of-word marker `</w>` appended.

`"low"` → `('l', 'o', 'w', '</w>')`

Why a tuple? Tuples are hashable — they can be dictionary keys. As BPE merges characters, the tuple grows shorter: `('l','o','w','</w>')` → `('lo','w','</w>')` → `('low','</w>')`.  
Why `</w>`? Without it, we cannot tell after merging whether `low` ends a word or is a prefix inside `lower`. The marker keeps word boundaries explicit throughout the algorithm.

In [7]:
# input : corpus string  e.g. 'low low lower newest'
# output: vocab = {('l','o','w','</w>'):2, ('l','o','w','e','r','</w>'):1, ...}
def get_vocab_from_corpus(corpus_str):
    # The function does 2 things:
    # 1. Split the given string to words and then count the frequency of each word.
    # 2. Split each word to chars followed by </w>

    # corpus_str.split() splits on whitespace — every word becomes one element
    # Counter turns that list into {word: count} e.g. {'low': 5, 'lower': 2, ...}
    word_counts = Counter(corpus_str.split())

    # create a empty dict
    vocab = {}

    # iterate over the word_count dict.
    for word, count in word_counts.items():
        # word  : a plain string,  e.g. 'low'
        # count : how many times it appears in the corpus, e.g. 5

        # list(word) splits the string into individual characters:
        #   'low'    → ['l', 'o', 'w']
        #   'newest' → ['n', 'e', 'w', 'e', 's', 't']
        #
        # + ['</w>'] appends the end-of-word marker:
        #   ['l', 'o', 'w'] + ['</w>'] → ['l', 'o', 'w', '</w>']
        #
        # tuple(...) converts to a tuple so it can be used as a dict key:
        #   ('l', 'o', 'w', '</w>')
        chars = tuple(list(word) + ['</w>'])

        # Store: key = char tuple, value = word frequency
        # This is what BPE will scan and modify in each merge step
        vocab[chars] = count

    return vocab


# Classic BPE toy corpus — small enough to trace by hand
# 5× 'low', 2× 'lower', 4× 'newest', 2× 'widest'
toy_corpus = "low low low low low lower lower newest newest newest newest widest widest"

vocab = get_vocab_from_corpus(toy_corpus)

print("Initial vocabulary (every word split into characters):")
print()
for word, freq in sorted(vocab.items(), key=lambda x: -x[1]):
    # Join with spaces so the individual tokens are easy to read
    print(f"  freq={freq}  →  {' '.join(word)}")

Initial vocabulary (every word split into characters):

  freq=5  →  l o w </w>
  freq=4  →  n e w e s t </w>
  freq=2  →  l o w e r </w>
  freq=2  →  w i d e s t </w>


### Step 2 — Count every adjacent pair

BPE's decision rule is simple: **merge whatever pair appears most often**.  
To find that pair, we scan every word and count every adjacent token pair — weighted by how often the word appears in the corpus.

If `'low'` appears 5 times, the pair `('l','o')` gets 5 counts from it alone.

In [8]:
# input : vocab dict  e.g. {('l','o','w','</w>'):5, ('l','o','w','e','r','</w>'):2}
# output: pairs = {('l','o'):7, ('o','w'):7, ('e','s'):6, ...}
def get_pairs(vocab):
    # pairs is a Counter — a dictionary that maps each (token_a, token_b) pair
    # to the total number of times that pair appears across the entire corpus.
    pairs = Counter()

    for word, freq in vocab.items():
        # 'word' is a tuple of tokens, e.g. ('l', 'o', 'w', '</w>')
        # 'freq' is how many times this word appears in the corpus, e.g. 5
        #
        # We walk every adjacent position in the tuple:
        #   i=0: pair = ('l', 'o')
        #   i=1: pair = ('o', 'w')
        #   i=2: pair = ('w', '</w>')
        # range(len(word) - 1) stops one before the end so word[i+1] is always valid
        for i in range(len(word) - 1):
            pair = (word[i], word[i+1])

            # Add freq, not 1 — a pair inside a word that appears 5 times
            # contributes 5 to the count, not 1.
            # This ensures common words drive the merge decisions.
            pairs[pair] += freq

    return pairs


# Count all pairs in the initial (character-level) vocabulary
pairs = get_pairs(vocab)

print("Top 10 adjacent pairs (sorted by total count in corpus):")
print()
print(f"  {'Pair':25s}  {'Count':>6}")
print("  " + "-" * 35)
for pair, count in pairs.most_common(10):
    print(f"  {str(pair):25s}  {count:>6}")

print()
# The pair with the highest count is the one BPE merges next
best_pair  = max(pairs, key=pairs.get)
best_count = pairs[best_pair]
print(f"  Winner: {best_pair}  (count: {best_count}) — this pair gets merged next")

Top 10 adjacent pairs (sorted by total count in corpus):

  Pair                        Count
  -----------------------------------
  ('l', 'o')                      7
  ('o', 'w')                      7
  ('w', 'e')                      6
  ('e', 's')                      6
  ('s', 't')                      6
  ('t', '</w>')                   6
  ('w', '</w>')                   5
  ('n', 'e')                      4
  ('e', 'w')                      4
  ('e', 'r')                      2

  Winner: ('l', 'o')  (count: 7) — this pair gets merged next


### Step 3 — Merge the winning pair

We now replace every occurrence of the winning pair with a single new token.  
The vocabulary shrinks by one token per word that contained that pair.  
The new token is added to the vocabulary — it can itself be part of a pair in the next round.

In [9]:
# input : pair + vocab  e.g. pair=('l','o'),  vocab={('l','o','w','</w>'):5, ...}
# output: updated vocab  e.g. {('lo','w','</w>'):5, ('lo','w','e','r','</w>'):2, ...}
# Remember, vocab has dict of : ('l','o','w','</w>') : 5 etc.
# pairs has dict of : ('l','o') : 7 etc.
#
# This function takes the winning pair and replaces every occurrence
# of that pair in the vocabulary with a single merged token.
#
# Example: pair = ('e', 's')
#
# Walking through 'newest' → ('n','e','w','e','s','t','</w>'):
#
#   i=0  word[0]='n'  word[1]='e'   → not ('e','s') → emit 'n',    i=1
#   i=1  word[1]='e'  word[2]='w'   → not ('e','s') → emit 'e',    i=2
#   i=2  word[2]='w'  word[3]='e'   → not ('e','s') → emit 'w',    i=3
#   i=3  word[3]='e'  word[4]='s'   → MATCH!        → emit 'es',   i=5  (skip 2)
#   i=5  word[5]='t'  word[6]='</w>'→ not ('e','s') → emit 't',    i=6
#   i=6  word[6]='</w>'             → not ('e','s') → emit '</w>', i=7
#
#   Result: ('n','e','w','es','t','</w>')   ← 7 tokens → 6 tokens
#
# Walking through 'low' → ('l','o','w','</w>'):
#
#   i=0  'l','o'     → not ('e','s') → emit 'l',    i=1
#   i=1  'o','w'     → not ('e','s') → emit 'o',    i=2
#   i=2  'w','</w>'  → not ('e','s') → emit 'w',    i=3
#   i=3  '</w>'      → not ('e','s') → emit '</w>', i=4
#
#   Result: ('l','o','w','</w>')   ← unchanged, no 'e'+'s' pair here
#
# Key insight: i += 2 is what makes this a merge, not just a rename.
# It physically skips the second token of the pair so it is never
# emitted on its own. Without i += 2, both 'es' and 's' would appear.

def merge_vocab(pair, vocab):
    # The new token is just the two strings concatenated
    # ('e', 's') → 'es'
    new_token = pair[0] + pair[1]
    new_vocab  = {}

    for word, freq in vocab.items():
        new_word = []
        i = 0
        while i < len(word):
            # Check whether the current position is the start of our target pair
            if i < len(word) - 1 and word[i] == pair[0] and word[i+1] == pair[1]:
                new_word.append(new_token)  # emit the merged token
                i += 2                       # skip past both original tokens
            else:
                new_word.append(word[i])     # emit this token unchanged
                i += 1                       # advance one position

        # Word frequency is unchanged — we only changed its internal representation
        new_vocab[tuple(new_word)] = freq

    return new_vocab


# Run one merge step manually so we can see the before/after
print(f"Merging pair: {best_pair}  →  '{best_pair[0] + best_pair[1]}'")
print()
print("Before merge:")
for word, freq in sorted(vocab.items(), key=lambda x: -x[1]):
    print(f"  freq={freq}  →  {' '.join(word)}")

vocab = merge_vocab(best_pair, vocab)

print()
print("After merge:")
for word, freq in sorted(vocab.items(), key=lambda x: -x[1]):
    print(f"  freq={freq}  →  {' '.join(word)}")

Merging pair: ('l', 'o')  →  'lo'

Before merge:
  freq=5  →  l o w </w>
  freq=4  →  n e w e s t </w>
  freq=2  →  l o w e r </w>
  freq=2  →  w i d e s t </w>

After merge:
  freq=5  →  lo w </w>
  freq=4  →  n e w e s t </w>
  freq=2  →  lo w e r </w>
  freq=2  →  w i d e s t </w>


In [10]:
# input : original vocab  e.g. {('l','o','w','</w>'):5, ...}
# output: 10-row merge table showing the true first 10 merges  e.g. step 1: l+o -> lo
# Run 10 BPE merge steps from the original unmodified vocabulary.
#
# Each iteration is one full BPE cycle:
#   1. get_pairs   → count all adjacent pairs in the current vocab
#   2. max(...)    → find the winning pair (highest count)
#   3. merge_vocab → apply the merge, update vocab

# Reset to original vocab so step 1 is genuinely the first merge.
vocab = get_vocab_from_corpus(toy_corpus)

print("BPE merge steps (toy corpus):")
print(f"  {'Step':>4}  {'Pair merged':>20}  {'Count':>6}  {'New token'}")
print("  " + "-"*55)

# create a list
merges = []

# run the loop 10 times
for step in range(10):
    # you get the adjacent pairs frequency.
    pairs = get_pairs(vocab)
    if not pairs:
        break
    # find the winning pair.
    best_pair = max(pairs, key=pairs.get)
    best_count = pairs[best_pair]
    new_token = best_pair[0] + best_pair[1]
    merges.append(best_pair)
    vocab = merge_vocab(best_pair, vocab)
    print(f"  {step+1:>4}  {repr(best_pair[0] + ' + ' + best_pair[1]):>20}  {best_count:>6}  → {repr(new_token)}")

print()
print("Final vocabulary:")
for word, freq in sorted(vocab.items(), key=lambda x: -x[1]):
    print(f"  {freq}× {' '.join(word)}")


BPE merge steps (toy corpus):
  Step           Pair merged   Count  New token
  -------------------------------------------------------
     1               'l + o'       7  → 'lo'
     2              'lo + w'       7  → 'low'
     3               'e + s'       6  → 'es'
     4              'es + t'       6  → 'est'
     5          'est + </w>'       6  → 'est</w>'
     6          'low + </w>'       5  → 'low</w>'
     7               'n + e'       4  → 'ne'
     8              'ne + w'       4  → 'new'
     9       'new + est</w>'       4  → 'newest</w>'
    10             'low + e'       2  → 'lowe'

Final vocabulary:
  5× low</w>
  4× newest</w>
  2× lowe r </w>
  2× w i d est</w>


### Why BPE Training Is Expensive

You just watched 10 merge steps on a 4-word toy corpus. Now imagine running 50,000 merges
on a corpus of 1 billion tokens.

Each merge step calls `get_pairs` and then `merge_vocab`, both of which walk every unique
word type and scan every adjacent token pair inside it:

```
  Cost per step  ≈  O(V × L)

  V  =  number of unique word types  (e.g. 25,000 for tinyshakespeare)
  L  =  average tokens per word      (starts at ~5 chars, shrinks as merges happen)
```

For N merge steps the total cost is roughly O(N × V × L) — this does not scale to
the large corpora used in practice.

**Our simple implementation rescans the entire vocabulary after every merge.**
Production BPE trainers maintain pair statistics incrementally and update only the
pairs adjacent to the newly merged token rather than recounting from scratch.
This makes large-scale tokenizer training practical without changing what the
algorithm learns.


**What you just watched:**
- Step 1 merged `l + o` → `lo` because `low` (×5) and `lower` (×2) both contain that pair — frequency 7
- Then `lo + w` → `low`, then word-endings like `est</w>` formed, then full words like `newest</w>` collapsed to one token
- BPE knows nothing about morphology — it only sees frequency. Frequently reused substrings sometimes align with morphemes, so morphological-looking units can emerge from the statistics alone.

BPE doesn't design for morphology. It finds it by accident, when morphemes happen to be the frequently reused parts.


---
## Section 5 — BPE on tinyshakespeare

Now scale up. We'll run 500 BPE merges on the actual corpus and watch the vocabulary and token count evolve.

In [11]:
# input : text corpus  e.g. 'First Citizen: Before...'
# output: bpe_vocab = {'l o w </w>':300, 'n e w e s t </w>':4, ...}  (25,670 entries)
# Build BPE on tinyshakespeare
#
# The toy section used tuple-keyed vocab:  ('l','o','w','</w>') : 5
# Here we switch to string-keyed vocab:    'l o w </w>'         : 5
#
# Why? The string representation enables a compact, boundary-safe regex
# merge implementation (see merge_vocab_str). The logic is identical to
# the tuple version — only the data structure changes.

def build_corpus_vocab(text):
    # Same as get_vocab_from_corpus() in the toy section, but stores each word
    # as a space-separated string instead of a tuple.
    #
    # 'low'    →  key = 'l o w </w>'
    # 'newest' →  key = 'n e w e s t </w>'
    #
    # Splitting on spaces later gives back the individual tokens,
    # so this representation is fully equivalent to the tuple version.

    word_counts = Counter(text.split())
    vocab = {}
    for word, count in word_counts.items():
        # list(word) → individual characters, + ['</w>'] → end-of-word marker
        # ' '.join(...) → space-separated token representation used by
        # the boundary-safe merge implementation below.
        key = ' '.join(list(word) + ['</w>'])
        vocab[key] = count
    return vocab


def get_pairs_str(vocab):
    # Same as get_pairs() in the toy section.
    # Split the string key back into a list of tokens, then walk adjacent pairs.
    #
    # Example: word_str = 'l o w </w>', freq = 5
    #   symbols = ['l', 'o', 'w', '</w>']
    #   pairs counted: ('l','o') += 5, ('o','w') += 5, ('w','</w>') += 5

    pairs = Counter()
    for word_str, freq in vocab.items():
        symbols = word_str.split()          # 'l o w </w>' → ['l','o','w','</w>']
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i+1])] += freq   # weight by word frequency
    return pairs


def merge_vocab_str(pair, vocab):
    # Same merge logic as the tuple version, using regex instead of str.replace().
    #
    # Why not plain str.replace()?
    # After several merges, tokens like 'fo' exist. The string 'fo r </w>'
    # contains the substring 'o r' -- str.replace would incorrectly match
    # inside the merged token 'fo', producing 'for </w>' instead of leaving
    # 'fo r </w>' unchanged. Confirmed bug: 63/100 merges diverge vs. the
    # tuple version and inflates the final vocabulary substantially.
    #
    # The fix: require token boundaries (space or start/end of string) on
    # both sides of the pair before matching.
    #
    # Example: pair = ('o', 'r')
    #   pattern = r'(?<![^\s])o r(?![^\s])'
    #   'fo r </w>'  -> no match  (o is inside merged token fo)
    #   'o r </w>'   -> 'or </w>'   ✓

    pattern     = r'(?<![^\s])' + re.escape(pair[0]) + r' ' + re.escape(pair[1]) + r'(?![^\s])'
    replacement = pair[0] + pair[1]
    new_vocab = {}
    for word_str, freq in vocab.items():
        new_vocab[re.sub(pattern, replacement, word_str)] = freq
    return new_vocab


print("Building BPE corpus vocabulary...")
bpe_vocab = build_corpus_vocab(text)
print(f"Unique word types: {len(bpe_vocab):,}")
print(f"Running 500 BPE merges...")


Building BPE corpus vocabulary...
Unique word types: 25,670
Running 500 BPE merges...


In [12]:
# ── Demonstration: what each function returns ────────────────────────────────
# Run on a tiny corpus so the data structures are visible before the 500-merge
# run on tinyshakespeare below.

demo_text  = "low low lower newer newest widest"
demo_vocab = build_corpus_vocab(demo_text)

print("1. build_corpus_vocab — each unique word → space-separated characters + </w>:")
for count, key in sorted((-v, k) for k, v in demo_vocab.items()):
    print(f"   {-count}×  {key!r}")

print()
demo_pairs = get_pairs_str(demo_vocab)
print("2. get_pairs_str — top 5 adjacent pairs (token1, token2) → frequency:")
for pair, count in demo_pairs.most_common(5):
    print(f"   {count}×  {pair}")

print()
best_pair  = max(demo_pairs, key=demo_pairs.get)
demo_merged = merge_vocab_str(best_pair, demo_vocab)
joined     = best_pair[0] + best_pair[1]
print(f"3. merge_vocab_str — apply most-frequent merge: {best_pair[0]!r} + {best_pair[1]!r}  →  {joined!r}")
print("   Before:"); [print(f"     {k!r}") for k in sorted(demo_vocab)]
print("   After: "); [print(f"     {k!r}") for k in sorted(demo_merged)]


1. build_corpus_vocab — each unique word → space-separated characters + </w>:
   2×  'l o w </w>'
   1×  'l o w e r </w>'
   1×  'n e w e r </w>'
   1×  'n e w e s t </w>'
   1×  'w i d e s t </w>'

2. get_pairs_str — top 5 adjacent pairs (token1, token2) → frequency:
   3×  ('l', 'o')
   3×  ('o', 'w')
   3×  ('w', 'e')
   2×  ('w', '</w>')
   2×  ('e', 'r')

3. merge_vocab_str — apply most-frequent merge: 'l' + 'o'  →  'lo'
   Before:
     'l o w </w>'
     'l o w e r </w>'
     'n e w e r </w>'
     'n e w e s t </w>'
     'w i d e s t </w>'
   After: 
     'lo w </w>'
     'lo w e r </w>'
     'n e w e r </w>'
     'n e w e s t </w>'
     'w i d e s t </w>'


[None, None, None, None, None]

### Training the tokenizer

The cell below runs the actual BPE training loop — 500 iterations of the same
three-step cycle demonstrated above, now on the full tinyshakespeare corpus.

The output is `bpe_merges`: an ordered list of 500 merge rules.
This is the trained tokenizer. Every downstream step in this chapter depends on it —
vocabulary construction, encoding, compression measurement, and the full Chapter 4B
scorecard all load and replay these rules.


In [13]:
# input : bpe_vocab = {'l o w </w>':300, ...}  (string-keyed, 25,670 word types)
# output: bpe_merges = [(('e','</w>'), 'e</w>'), (('t','h'), 'th'), ...]  500 rules
# Run 500 BPE merge steps on the full tinyshakespeare corpus.
#
# Same 3-step cycle as the toy section, now at scale:
#   1. get_pairs_str   -> count all adjacent pairs in bpe_vocab
#   2. max(...)        -> pick the winning pair (highest count)
#   3. merge_vocab_str -> apply the merge, update bpe_vocab
#
# bpe_merges records every merge in order: list of (pair, new_token).
# This list is what the encoder replays at inference time --
# to tokenize a new word, apply all 500 merges in the same order.
#
# merge_log is sparse: first 10 steps + every 50th.
# Early steps: single chars fuse    ('e','</w>') -> 'e</w>', ('t','h') -> 'th'
# Middle steps: subwords emerge     'th' + 'e'   -> 'the',  'i' + 'n' -> 'in'
# Late steps:   rare proper nouns   'ARD</w>' (from RICHARD), 'VINC' (VINCENTIO)
NUM_MERGES = 500
bpe_merges = []      # list of (pair, new_token) -- replayed by the encoder
merge_log = []       # sparse sample for display only

for step in range(NUM_MERGES):
    pairs = get_pairs_str(bpe_vocab)
    if not pairs:
        break
    best = max(pairs, key=pairs.get)    # winning pair -- highest frequency
    best_count = pairs[best]
    new_token = best[0] + best[1]       # concatenate to form the merged token
    bpe_merges.append((best, new_token))
    bpe_vocab = merge_vocab_str(best, bpe_vocab)   # update vocab in-place

    # Log: first 10 steps + every 50th
    if (step + 1) % 50 == 0 or step < 10:
        merge_log.append((step + 1, new_token, best_count))

print(f"Completed {len(bpe_merges)} merges.")
print()
print("Sample merges (step, new token, pair frequency):")
print(f"  {'Step':>5}  {'New token':>18}  {'Pair freq':>10}")
print("  " + "-"*40)
for step, tok, count in merge_log:
    print(f"  {step:>5}  {repr(tok):>18}  {count:>10,}")

Completed 500 merges.

Sample merges (step, new token, pair frequency):
   Step           New token   Pair freq
  ----------------------------------------
      1             'e</w>'      29,077
      2                'th'      22,739
      3             ',</w>'      19,603
      4             't</w>'      17,300
      5             's</w>'      16,256
      6             'd</w>'      14,957
      7                'ou'      12,730
      8                'er'      11,771
      9             'y</w>'      10,679
     10                'in'      10,606
     50                'se'       2,489
    100            'O:</w>'       1,494
    150                'me'         848
    200                'ET'         628
    250           'was</w>'         492
    300               'der'         392
    350           'end</w>'         318
    400                'ey'         273
    450          'most</w>'         242
    500             'QUEEN'         216


### Building the vocabulary

`bpe_merges` gives us 500 merge rules, but not yet a lookup table.
The next step converts those rules into two dictionaries — `stoi_bpe` (token → integer ID)
and `itos_bpe` (integer ID → token) — that the encoder and decoder will use.

One subtlety: the vocabulary must include **every merge product**, not just tokens that
survive in the final corpus. A merge like `t + h → th` may later be fully absorbed
into `th + e → the`, making `'th'` disappear from the corpus — but the encoder can
still emit `'th'` when it tokenizes an unseen word. Missing it causes a KeyError.


In [14]:
# input : bpe_vocab after 500 merges, bpe_merges list, text corpus
# output: stoi_bpe = {'a':0, 'b':1, ..., 'the</w>':500},  itos_bpe = {0:'a', ...}
#
# Important: build the vocabulary from (a) every base character symbol and (b) every
# merge product -- NOT just tokens that survive in the final corpus.
#
# Why: a merge like  t + h -> th  may be entirely absorbed into later merges on the
# training corpus (th + e -> the), so 'th' disappears from bpe_vocab.keys().
# But tokenize_word_bpe() can still emit 'th' on a new, unseen word.
# If 'th' is missing from stoi_bpe the encoder crashes.
#
# Correct vocabulary = base alphabet (chars + </w>) + every learned merge symbol.

base_bpe_tokens = {ch for word in text.split() for ch in word}
base_bpe_tokens.add("</w>")

learned_bpe_tokens = {new_token for _, new_token in bpe_merges}

bpe_token_set = base_bpe_tokens | learned_bpe_tokens

# Sort: single characters first, then longer tokens alphabetically.
bpe_tokens = sorted(bpe_token_set, key=lambda x: (len(x), x))
stoi_bpe = {tok: i for i, tok in enumerate(bpe_tokens)}
itos_bpe = {i: tok for i, tok in enumerate(bpe_tokens)}

print(f"BPE vocabulary size  : {len(bpe_tokens):,} tokens")
print(f"  base symbols       : {len(base_bpe_tokens)}  ({len(base_bpe_tokens)-1} corpus chars + </w>)")
print(f"  learned merges     : {len(learned_bpe_tokens)}")
print()

long_toks = [t for t in bpe_tokens if '</w>' in t and len(t.replace('</w>', '')) >= 4][:20]
print("Sample multi-char tokens (complete words, with </w>):")
for t in long_toks:
    print(f"  {t.replace('</w>', '')!r}")


BPE vocabulary size  : 564 tokens
  base symbols       : 64  (63 corpus chars + </w>)
  learned merges     : 500

Sample multi-char tokens (complete words, with </w>):
  'DUKE'
  "I'll"
  'IUS:'
  'KING'
  'TIO:'
  'That'
  'This'
  'What'
  'With'
  'come'
  'ence'
  'fore'
  'from'
  'give'
  'good'
  'hath'
  'have'
  'here'
  'hich'
  'irst'


### The tokenizer interface

With the vocabulary in place, we can define the full tokenizer interface:
`tokenize_word_bpe` replays the 500 learned rules on a single word;
`encode_bpe` splits a string into words and maps each to integer IDs;
`decode_bpe` converts IDs back to text by stripping the `</w>` markers.

This cell also measures the **token count on the full corpus** — the raw number we
need to compute the compression ratio in the next step.


In [15]:
# input : word + bpe_merges  e.g. word='lowest',  merges=[(('l','o'),'lo'), ...]
# output: token list  e.g. tokenize_word_bpe('lowest') -> ['low', 'est</w>']
# Three functions that together form the BPE tokenizer interface:
#   tokenize_word_bpe -- apply learned merges to one word  -> list of token strings
#   encode_bpe        -- tokenize a full string            -> list of integer IDs
#   decode_bpe        -- convert integer IDs back to text  -> string

def tokenize_word_bpe(word, merges):
    # Start: split word into individual characters and append end-of-word marker.
    # 'low' -> ['l', 'o', 'w', '</w>']
    symbols = list(word) + ['</w>']

    # Replay every merge rule in training order.
    # Each rule fuses one specific pair wherever it appears in symbols.
    # After 500 rules, common subwords have been collapsed to single strings.
    for pair, new_token in merges:
        i = 0
        new_symbols = []
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == pair[0] and symbols[i+1] == pair[1]:
                new_symbols.append(new_token)   # fuse pair into one token
                i += 2                           # skip both originals
            else:
                new_symbols.append(symbols[i])  # keep unchanged
                i += 1
        symbols = new_symbols
    return symbols   # e.g. ['low</w>'] after all merges


def encode_bpe(text, merges, stoi):
    # Split on whitespace, tokenize each word, map tokens -> integer IDs.
    # With vocabulary built from base + merge products, every token emitted
    # by tokenize_word_bpe() is guaranteed to exist in stoi. Hard-fail if not.
    ids = []
    for word in text.split():
        tokens = tokenize_word_bpe(word, merges)
        ids.extend(stoi[t] for t in tokens)
    return ids


def decode_bpe(ids, itos):
    # Replace the </w> marker with a space -- this reconstructs word boundaries.
    # 'low</w>' -> 'low ', 'er</w>' -> 'er '  -> joined: 'lower '  -> stripped: 'lower'
    return ''.join(itos[i].replace('</w>', ' ') for i in ids).strip()


# Measure token count on full corpus to compare compression.
# tokens_char has one token per character (ratio = 1.0 by definition).
token_count_bpe = sum(
    len(tokenize_word_bpe(w, bpe_merges))
    for w in text.split()
)
# Use normalized_text length for a fair apples-to-apples comparison.
# Both tokenizers process the same whitespace-collapsed character stream.
ratio = token_count_bpe / len(normalized_text)

print("Token count comparison:")
print(f"  Character tokenizer : {len(normalized_text):>12,} tokens  (vocab: {vocab_size_char})")
print(f"    (1 token per char of normalized_text)")
print(f"  BPE (500 merges)    : {token_count_bpe:>12,} tokens  (vocab: {len(bpe_tokens)})")
print(f"  Reduction           : {ratio:.3f}x  ({(1-ratio)*100:.1f}% fewer tokens)")
print(f"  Avg chars per BPE token: {len(normalized_text)/token_count_bpe:.2f}  (vs normalized text)")


Token count comparison:
  Character tokenizer :    1,108,152 tokens  (vocab: 65)
    (1 token per char of normalized_text)
  BPE (500 merges)    :      461,460 tokens  (vocab: 564)
  Reduction           : 0.416x  (58.4% fewer tokens)
  Avg chars per BPE token: 2.40  (vs normalized text)


### Scorecard — Measurement 1: compression

The compression ratio answers: *how many tokens does each tokenizer need per character of text?*

Both ratios are measured against `normalized_text` (whitespace-collapsed) so the denominator
is identical for both tokenizers. The result goes into the scorecard that Chapter 4B
will use to compare tokenizer quality across all three dimensions.


In [16]:
# input : token counts  e.g. tokens_char=1,115,394,  token_count_bpe=461,460
# output: scorecard['bpe_500']['compression'] = <measured value>  (tokens/char)
# Record Measurement 1 in the scorecard: compression ratio (tokens / chars).
# Both ratios use normalized_text so both tokenizers are measured on the same text length.
# The character tokenizer is re-run on normalized_text to get a fair count.
char_tokens_normalized = encode_char(normalized_text)
char_ratio = len(char_tokens_normalized) / len(normalized_text)  # should be exactly 1.0
bpe_ratio  = token_count_bpe            / len(normalized_text)

scorecard["char_tokenizer"]["compression"] = round(char_ratio, 3)
scorecard["bpe_500"]["compression"] = round(bpe_ratio, 3)

print("Compression scores (tokens/char -- lower = better, vs normalized text):")
print(f"  Char tokenizer : {char_ratio:.3f}  ({len(char_tokens_normalized):,} tokens / {len(normalized_text):,} chars)")
print(f"  BPE 500 merges : {bpe_ratio:.3f}  ({token_count_bpe:,} tokens / {len(normalized_text):,} chars)")
print(f"  BPE is {char_ratio/bpe_ratio:.1f}x more compressed")


Compression scores (tokens/char -- lower = better, vs normalized text):
  Char tokenizer : 1.000  (1,108,152 tokens / 1,108,152 chars)
  BPE 500 merges : 0.416  (461,460 tokens / 1,108,152 chars)
  BPE is 2.4x more compressed


---
## Section 6 — Encode and Decode

A production tokenizer **should be lossless**: encoding and then decoding should recover the original text exactly. Let's test whether ours is.


In [17]:
# input : sentence  e.g. 'To be or not to be'
# output: BPE token display  e.g. ['To↵', 'be↵', 'or↵', 'not↵', 'to↵', 'be↵']  (6 tokens)
# Show what BPE tokenization looks like on three familiar sentences.
# For each sentence:
#   1. tokenize_word_bpe splits every word into BPE tokens
#   2. </w> is replaced with the display symbol ↵ so word boundaries are visible
#   3. BPE token count is compared with the character-token count for the same sentence.
test_sentences = [
    "To be or not to be",
    "What a piece of work is man",
    "All the world's a stage",
]

print("BPE tokenization examples (500 merges):")
print()
for sentence in test_sentences:
    token_strings = []
    for word in sentence.split():
        toks = tokenize_word_bpe(word, bpe_merges)
        token_strings.extend(toks)

    # Display: replace </w> marker with ↵ for readability
    display_toks = [t.replace('</w>', '↵') for t in token_strings]

    print(f"  Input  : {sentence!r}")
    print(f"  Tokens : {display_toks}")
    print(f"  Count  : {len(token_strings)} BPE tokens  vs  {len(encode_char(sentence))} char tokens")
    print()


BPE tokenization examples (500 merges):

  Input  : 'To be or not to be'
  Tokens : ['To↵', 'be↵', 'or↵', 'not↵', 'to↵', 'be↵']
  Count  : 6 BPE tokens  vs  18 char tokens

  Input  : 'What a piece of work is man'
  Tokens : ['What↵', 'a↵', 'p', 'i', 'e', 'ce↵', 'of↵', 'wor', 'k↵', 'is↵', 'man↵']
  Count  : 11 BPE tokens  vs  27 char tokens

  Input  : "All the world's a stage"
  Tokens : ['A', 'll↵', 'the↵', 'wor', 'l', 'd', "'s↵", 'a↵', 'sta', 'ge↵']
  Count  : 10 BPE tokens  vs  23 char tokens



In [18]:
# input : sentence  e.g. "All the world's a stage..."
# output: Round-trip PASS on single-spaced text, FAIL on multi-line text
# Test 1: single-spaced sentence -- should pass
sentence = "All the world's a stage and all the men and women merely players"
ids_rt = encode_bpe(sentence, bpe_merges, stoi_bpe)
recovered = decode_bpe(ids_rt, itos_bpe)
print(f"Test 1 -- single-spaced sentence")
print(f"  Original  : {sentence!r}")
print(f"  Decoded   : {recovered!r}")
print(f"  Round-trip: {'PASS ✓' if recovered == sentence else 'FAIL ✗'}")
print()

# Test 2: text with internal newlines -- deliberately shows the limitation
sentence2 = "To be\n\nor not to be"
ids_rt2 = encode_bpe(sentence2, bpe_merges, stoi_bpe)
recovered2 = decode_bpe(ids_rt2, itos_bpe)
print(f"Test 2 -- text with newlines")
print(f"  Original  : {sentence2!r}")
print(f"  Decoded   : {recovered2!r}")
print(f"  Round-trip: {'PASS ✓' if recovered2 == sentence2 else 'FAIL ✗  <-- whitespace lost'}")
print()
print("Why? encode_bpe calls text.split(), which collapses all whitespace --")
print("newlines, tabs, and repeated spaces all become single word separators.")
print()
print("This educational tokenizer preserves normalized word boundaries, not exact whitespace.")
print()
print("Production tokenizers approach this differently:")
print("  GPT-2   byte-level BPE: every byte has its own base token, including newlines.")
print("  SentencePiece: represents word-initial spaces explicitly (▁) rather than")
print("  discarding them during a whitespace-splitting step.")


Test 1 -- single-spaced sentence
  Original  : "All the world's a stage and all the men and women merely players"
  Decoded   : "All the world's a stage and all the men and women merely players"
  Round-trip: PASS ✓

Test 2 -- text with newlines
  Original  : 'To be\n\nor not to be'
  Decoded   : 'To be or not to be'
  Round-trip: FAIL ✗  <-- whitespace lost

Why? encode_bpe calls text.split(), which collapses all whitespace --
newlines, tabs, and repeated spaces all become single word separators.

This educational tokenizer preserves normalized word boundaries, not exact whitespace.

Production tokenizers approach this differently:
  GPT-2   byte-level BPE: every byte has its own base token, including newlines.
  SentencePiece: represents word-initial spaces explicitly (▁) rather than
  discarding them during a whitespace-splitting step.


---

## What we built

```
  Raw text
     ↓  build_corpus_vocab
  {word: freq} with </w> markers
     ↓  get_pairs → max → merge_vocab  x500
  500 merge rules
     ↓  encode_bpe
  token IDs
     ↓  decode_bpe
  normalized single-spaced text
```

We have a working word-boundary BPE tokenizer, with one known limitation:
our pre-tokenizer splits on whitespace, so newlines and repeated spaces are lost
in the round-trip. Section 6 demonstrates this deliberately.
Production tokenizers avoid this simple whitespace-splitting limitation in different
ways: byte-level BPE represents bytes directly, while SentencePiece can represent
whitespace explicitly.

**Chapter 4B** asks what we sacrificed to get here:
what structure did BPE destroy, can the model recover it,
and how do we measure whether the tradeoff was worth it?


---
## PyTorch Lab — Chapter 4A

Three exercises on the mechanics of tokenization.
All use the tinyshakespeare corpus and the BPE tokenizer built above.


In [19]:
# input : text corpus
# output: unique character count vs unique word count  e.g. 65 chars, 25,670 words
# Exercise 1 — Vocabulary size comparison
# Count unique characters vs unique words in tinyshakespeare.
# What is the ratio? What does this tell you about vocabulary explosion?

unique_chars = set(text)
unique_words_ex = set(text.split())

print("Exercise 1: Vocabulary sizes")
print(f"  Unique characters : {len(unique_chars):,}")
print(f"  Unique words      : {len(unique_words_ex):,}")
print(f"  Ratio             : {len(unique_words_ex)/len(unique_chars):.1f}× more word types than char types")
# → What does this ratio mean for embedding table size at word vs char level?

Exercise 1: Vocabulary sizes
  Unique characters : 65
  Unique words      : 25,670
  Ratio             : 394.9× more word types than char types


In [20]:
# input : sentence 'To be or not to be that is the question'
# output: token counts for char / BPE-100 / BPE-500  e.g. 31 / 14 / 10 tokens
# Exercise 2 — Token count comparison on a known sentence
# Encode "To be or not to be, that is the question" with:
# (a) character tokenizer, (b) BPE(100 merges), (c) BPE(500 merges)
# Report token counts and show the token strings.

sentence_ex3 = "To be or not to be that is the question"

print("Exercise 2: Token count comparison")
print(f"  Sentence: {sentence_ex3!r}")
print()

# (a) character tokenizer
ids_char = encode_char(sentence_ex3)
print(f"  (a) Char tokenizer  : {len(ids_char)} tokens")
print(f"      {list(sentence_ex3[:20])}...")

# (b) BPE 100 merges
merges_100 = bpe_merges[:100]
toks_100 = []
for w in sentence_ex3.split():
    toks_100.extend(tokenize_word_bpe(w, merges_100))
print(f"  (b) BPE(100 merges) : {len(toks_100)} tokens")
print(f"      {[t.replace('</w>','') for t in toks_100]}")

# (c) BPE 500 merges
toks_500 = []
for w in sentence_ex3.split():
    toks_500.extend(tokenize_word_bpe(w, bpe_merges))
print(f"  (c) BPE(500 merges) : {len(toks_500)} tokens")
print(f"      {[t.replace('</w>','') for t in toks_500]}")


Exercise 2: Token count comparison
  Sentence: 'To be or not to be that is the question'

  (a) Char tokenizer  : 39 tokens
      ['T', 'o', ' ', 'b', 'e', ' ', 'o', 'r', ' ', 'n', 'o', 't', ' ', 't', 'o', ' ', 'b', 'e', ' ', 't']...
  (b) BPE(100 merges) : 15 tokens
      ['T', 'o', 'be', 'or', 'not', 'to', 'be', 'that', 'is', 'the', 'q', 'u', 'es', 'ti', 'on']
  (c) BPE(500 merges) : 12 tokens
      ['To', 'be', 'or', 'not', 'to', 'be', 'that', 'is', 'the', 'qu', 'es', 'tion']


In [21]:
# input : passage  e.g. 'What a piece of work is man'
# output: token IDs, decoded string, re-encoded IDs match = True
# Exercise 3 — Round-trip decode
# Given these token IDs from our BPE(500) tokenizer, decode them to text.
# Then re-encode the decoded text and verify the IDs match.

print("Exercise 3: Round-trip verification")
print()

# Encode a passage to get the IDs, then pretend we only have the IDs
passage = "What a piece of work is man how noble in reason"
original_ids = encode_bpe(passage, bpe_merges, stoi_bpe)

print(f"  Original  : {passage!r}")
print(f"  Token IDs : {original_ids}")
print()

# Now decode
decoded = decode_bpe(original_ids, itos_bpe)
print(f"  Decoded   : {decoded!r}")

# Re-encode
reencoded = encode_bpe(decoded, bpe_merges, stoi_bpe)
match = original_ids == reencoded
print(f"  Re-encoded IDs match: {match}")
print()
if not match:
    print("  This educational tokenizer preserves normalized word boundaries,")
    print("  not exact whitespace. Re-encode of decoded text always succeeds")
    print("  because both encode and decode operate on the same normalized form.")


Exercise 3: Round-trip verification

  Original  : 'What a piece of work is man how noble in reason'
  Token IDs : [503, 310, 52, 45, 41, 352, 391, 278, 315, 378, 462, 458, 159, 435, 376, 261, 55, 393]

  Decoded   : 'What a piece of work is man how noble in reason'
  Re-encoded IDs match: True

